# Import

In [ ]:
from pathlib import Path
import requests
import gzip
import xml.etree.ElementTree as ET
import json
import time
from tqdm.notebook import tqdm

In [ ]:
# Dictionnaire associant le nom du média (utilisé comme identifiant de fichier) 
# à l'URL de son sitemap d'index ou à une liste de sous-sitemaps explicites.
media = {
    "le_figaro": "https://sitemaps.lefigaro.fr/lefigaro.fr/articles.xml",
    "le_capital": "https://www.capital.fr/sitemap/articles.xml",
    "nice_matin": "https://www.nicematin.com/sitemap.xml",
    "le_nouvel_observateur": "https://www.nouvelobs.com/sitemap/sitemap-index-articles.xml",
    "le_journal_du_dimanche": "https://www.lejdd.fr/sitemap.xml",
    "le_monde": "https://www.lemonde.fr/sitemap_index.xml",
    "les_echos": "https://sitemap.lesechos.fr/sitemap_index.xml",
    "telerama": "https://www.telerama.fr/sitemaps/sitemap_index.php",
    "valeurs_actuelles": [
        "https://www.valeursactuelles.com/post-sitemap.xml",
        "https://www.valeursactuelles.com/post-sitemap2.xml",
        "https://www.valeursactuelles.com/post-sitemap3.xml",
    ],
}

# Liste d'URLs de sitemaps à ignorer (ex: sitemaps de catégories ou ressources non pertinentes).
to_ignore = {
    "https://www.nicematin.com/sitemap_fixed.xml",
    "https://www.nouvelobs.com/sitemap/sitemap-index-categories.xml",
}


## Partie 2

In [ ]:
media = {"l_opinion": "https://www.lopinion.fr/sitemap.xml",
         "challenges": "https://www.challenges.fr/sitemap.xml",
         "voix_du_nord" : "https://www.lavoixdunord.fr/sites/default/files/sitemaps/www_lavoixdunord_fr/sitemapindex.xml",
         "sud_ouest" : "https://www.sudouest.fr/sitemap.xml"
         }

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/126.0.0.0 Safari/537.36"
}

to_ignore = None

In [ ]:
# Répertoire de destination pour la sauvegarde des données extraites.
OUTPUT_DIR = Path("../data/urls")

# Télécharge et retourne le contenu d'un fichier XML. Gère la décompression automatique si le format est GZIP.
def read_xml(url_cible):
    raw_content = requests.get(url_cible, headers=HEADERS, timeout=60).content
    
    if not raw_content:
        raise ValueError("Contenu vide retourné par la requête.")

    # Décompression si le contenu est au format GZIP    
    if raw_content[:2] == b'\x1f\x8b':
        raw_content = gzip.decompress(raw_content)
        
    return raw_content


for media_name, sitemap in tqdm(media.items(), desc="Médias", unit="média"):

    # Création du fichier de sortie pour le média courant
    DATA_FILE = OUTPUT_DIR / f"{media_name}_articles.jsonl"
    
    # Initialisation avec les sitemaps à ignorer
    checked_sitemap = set(to_ignore or ())

    # Chargement de l'historique des traitements si le fichier de sortie existe déjà.
    if DATA_FILE.exists():
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            for line in f:
                checked_sitemap.add(json.loads(line)["sitemap"])

    # Si la source est une liste, on l'utilise directement comme liste de sous-sitemaps.
    if isinstance(sitemap, list):
        sub_sitemaps = sitemap
    
    # Sinon, on télécharge le sitemap d'index pour en extraire la liste des sous-sitemaps.
    else:
        try:
            raw_xml = read_xml(sitemap)
            root = ET.fromstring(raw_xml)
            sub_sitemaps = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
        except Exception:
            # L'index n'a pas pu être lu/parsé : on marque le média en échec et on passe au suivant.
            with open(DATA_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps({"type": "echec", "sitemap": sitemap}) + "\n")
            continue

    for sub_sitemap in tqdm(sub_sitemaps, desc=f"{media_name}", unit="sitemap", leave=False):
        
        if sub_sitemap in checked_sitemap:
            continue

        try:
            raw_xml = read_xml(sub_sitemap)
            root = ET.fromstring(raw_xml)
            urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
            result = {"type": "ok", "sitemap": sub_sitemap, "urls": urls}
        except Exception:
            result = {"type": "echec", "sitemap": sub_sitemap}

        # Sauvegarde et marquage en une seule étape
        with open(DATA_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(result) + "\n")
        checked_sitemap.add(sub_sitemap)

        time.sleep(1)

# L'Express

L'Express n'expose pas de sitemap d'index : on génère nous-mêmes la liste des sous-sitemaps, un par mois, de la forme `weeks/{année}-{mois}`. Chaque sous-sitemap contient directement les URLs d'articles, donc l'extraction est la même que pour les autres médias. La boucle couvre 1990-01 → 2026-06 et reprend là où elle s'est arrêtée grâce au fichier `.jsonl`.

In [ ]:
OUTPUT_DIR = Path("../data/urls")
DATA_FILE = OUTPUT_DIR / "l_express_articles.jsonl"

periodes = [(y, m) for y in range(1990, 2027) for m in range(1, 13) if not (y == 2026 and m > 6)]

checked_sitemap = set()
if DATA_FILE.exists():
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        for line in f:
            checked_sitemap.add(json.loads(line)["sitemap"])

for (year, month) in tqdm(periodes, desc="l_express", unit="sitemap"):
    sub_sitemap = f"https://www.lexpress.fr/arc/outboundfeeds/sitemap-all/weeks/{year}-{month}/?outputType=xml"

    if sub_sitemap in checked_sitemap:
        continue

    try:
        raw_xml = read_xml(sub_sitemap)
        root = ET.fromstring(raw_xml)
        urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
        result = {"type": "ok", "sitemap": sub_sitemap, "urls": urls}
    except Exception:
        result = {"type": "echec", "sitemap": sub_sitemap}

    with open(DATA_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(result) + "\n")
    checked_sitemap.add(sub_sitemap)

    time.sleep(1)


In [ ]:
from curl_cffi import requests as cffi_requests
import gzip
import xml.etree.ElementTree as ET

url = "https://www.lavoixdunord.fr/sites/default/files/sitemaps/www_lavoixdunord_fr/sitemapindex.xml"

# curl_cffi imite la signature TLS d'un vrai Chrome -> contourne Akamai Bot Manager.
reponse = cffi_requests.get(url, impersonate="chrome", timeout=60)
print("Code HTTP :", reponse.status_code)
print("Content-Type :", reponse.headers.get("Content-Type"))
print("Taille du contenu :", len(reponse.content), "octets")
print("URL finale (après redirections) :", reponse.url)

raw = reponse.content

if raw[:2] == b'\x1f\x8b':
    print("\n-> Contenu GZIP détecté, décompression...")
    raw = gzip.decompress(raw)

print("\n--- 500 premiers caractères ---")
print(raw[:500].decode("utf-8", errors="replace"))

print("\n--- Test du parsing XML ---")
try:
    root = ET.fromstring(raw)
    print("XML OK, balise racine :", root.tag)
except ET.ParseError as e:
    print("ParseError :", e)


# La Voix du Nord

La Voix du Nord est protégée par Akamai Bot Manager, qui inspecte la signature TLS de la connexion : `requests` est bloqué (403) quel que soit le User-Agent. On passe donc par `curl_cffi`, qui imite la signature d'un vrai Chrome. La structure reste classique (un sitemap d'index → des sous-sitemaps), seule la fonction de téléchargement change. La boucle reprend là où elle s'est arrêtée grâce au fichier `.jsonl`.

In [ ]:
from curl_cffi import requests as cffi_requests

OUTPUT_DIR = Path("../data/urls")
DATA_FILE = OUTPUT_DIR / "voix_du_nord_articles.jsonl"
INDEX = "https://www.lavoixdunord.fr/sites/default/files/sitemaps/www_lavoixdunord_fr/sitemapindex.xml"
NS = "{http://www.sitemaps.org/schemas/sitemap/0.9}"


# Téléchargement via curl_cffi (imitation Chrome) pour contourner Akamai.
def read_xml_cffi(url_cible):
    raw_content = cffi_requests.get(url_cible, impersonate="chrome", timeout=60).content
    if not raw_content:
        raise ValueError("Contenu vide retourné par la requête.")
    if raw_content[:2] == b'\x1f\x8b':
        raw_content = gzip.decompress(raw_content)
    return raw_content


# Téléchargement de l'index pour obtenir la liste des sous-sitemaps.
root = ET.fromstring(read_xml_cffi(INDEX))
sub_sitemaps = [loc.text for loc in root.findall(f".//{NS}loc")]

checked_sitemap = set()
if DATA_FILE.exists():
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        for line in f:
            checked_sitemap.add(json.loads(line)["sitemap"])

for sub_sitemap in tqdm(sub_sitemaps, desc="voix_du_nord", unit="sitemap"):

    if sub_sitemap in checked_sitemap:
        continue

    try:
        raw_xml = read_xml_cffi(sub_sitemap)
        root = ET.fromstring(raw_xml)
        urls = [loc.text for loc in root.findall(f".//{NS}loc")]
        result = {"type": "ok", "sitemap": sub_sitemap, "urls": urls}
    except Exception:
        result = {"type": "echec", "sitemap": sub_sitemap}

    with open(DATA_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(result) + "\n")
    checked_sitemap.add(sub_sitemap)

    time.sleep(1)


In [ ]:
# Compte les URLs (entrées "ok") de chaque .jsonl de data/urls, et le total.
OUTPUT_DIR = Path("../data/urls")

total = 0
for fichier in sorted(OUTPUT_DIR.glob("*_articles.jsonl")):
    n = 0
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            entree = json.loads(ligne)
            if entree["type"] == "ok":
                n += len(entree["urls"])
    print(f"{fichier.stem.removesuffix('_articles'):<20} {n:>10,} URLs")
    total += n

print(f"{'TOTAL':<20} {total:>10,} URLs")
